<a href="https://colab.research.google.com/github/anushka24-student/Real-Time-Log-Monitoring-Tool/blob/main/Real_Time_Log_Monitoring_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y golang-go
!go version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  golang-1.22-go golang-1.22-src golang-src
Suggested packages:
  bzr | brz mercurial subversion
The following NEW packages will be installed:
  golang-1.22-go golang-1.22-src golang-go golang-src
0 upgraded, 4 newly installed, 0 to remove and 6 not upgraded.
Need to get 45.7 MB of archives.
After this operation, 228 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 golang-1.22-src all 1.22.2-2ubuntu0.5 [19.7 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 golang-1.22-go amd64 1.22.2-2ubuntu0.5 [26.0 MB]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 golang-src all 2:1.

In [2]:
import os

os.makedirs("sample-logs", exist_ok=True)

with open("sample-logs/app.log", "w") as f:
    f.write("""INFO: Application started
ERROR: Database connection failed
INFO: User logged in
WARNING: High memory usage
CRITICAL: Server overloaded
ERROR: Request timeout
""")

with open("sample-logs/server.log", "w") as f:
    f.write("""INFO: Server running
ERROR: Disk space low
INFO: Request received
FAILURE: Authentication failed
ERROR: Service unavailable
""")

print("Sample log files created successfully!")

Sample log files created successfully!


In [6]:
%%writefile main.go
package main

import (
	"bufio"
	"fmt"
	"log"
	"os"
	"os/signal"
	"path/filepath"
	"strings"
	"sync"
	"syscall"
	"time"
)

// LogEvent represents a structured log event
type LogEvent struct {
	Timestamp string
	Level     string
	Source    string
	Message   string
}

// Monitor manages log watching and synchronization
type Monitor struct {
	events chan LogEvent
	wg     sync.WaitGroup
}

// watchFile monitors a log file for new lines.
func (m *Monitor) watchFile(path string, done chan struct{}) {
	defer m.wg.Done()

	filename := filepath.Base(path)
	file, err := os.Open(path)
	if err != nil {
		log.Printf("Error opening file %s: %v", path, err)
		return
	}
	defer file.Close()

	log.Printf("Monitoring: %s", path)
	reader := bufio.NewReader(file)

	for {
		select {
		case <-done:
			return
		default:
			line, err := reader.ReadString('\n')
			if err != nil {
				time.Sleep(500 * time.Millisecond) // Wait for new logs to be written
				continue
			}

			line = strings.TrimSpace(line)
			if line == "" {
				continue
			}

			// Extract log level (INFO, WARNING, ERROR, CRITICAL, FAILURE)
			level := "INFO"
			parts := strings.SplitN(line, ":", 2)
			if len(parts) > 1 {
				level = strings.TrimSpace(parts[0])
			}

			m.events <- LogEvent{
				Timestamp: time.Now().Format("15:04:05"),
				Level:     level,
				Source:    filename,
				Message:   line,
			}
		}
	}
}

// processEvents listens on the events channel and formats output
func processEvents(events chan LogEvent, done chan struct{}) {
	for {
		select {
		case event, ok := <-events:
			if !ok {
				return
			}
			fmt.Printf("\n[%s] %s | %s | %s\n", event.Timestamp, event.Level, event.Source, event.Message)
		case <-done:
			return
		}
	}
}

func main() {
	// Files to monitor (missing.log ko hata diya gaya hai)
	files := []string{
		"sample-logs/app.log",
		"sample-logs/server.log",
	}

	monitor := &Monitor{
		events: make(chan LogEvent, 100),
	}
	done := make(chan struct{})

	// Start event processor
	go processEvents(monitor.events, done)

	// Start watcher goroutines for each log file
	for _, path := range files {
		monitor.wg.Add(1)
		go monitor.watchFile(path, done)
	}

	fmt.Println("===================================")
	fmt.Println("      LOGPULSE MONITOR STARTED")
	fmt.Println("===================================")
	fmt.Println("Monitoring multiple log files...")
	fmt.Println("Press Ctrl+C to stop and generate reports.")

	// Wait for Ctrl+C signal
	signalChannel := make(chan os.Signal, 1)
	signal.Notify(signalChannel, os.Interrupt, syscall.SIGTERM)
	<-signalChannel

	fmt.Println("\nStopping LogPulse...")

	// Signal goroutines to stop and wait for worker completion
	close(done)
	monitor.wg.Wait()

	// Safely close the channel after all workers finish
	close(monitor.events)

	fmt.Println("LogPulse stopped.")
}

Overwriting main.go


In [7]:
!go run main.go

      LOGPULSE MONITOR STARTED
Monitoring multiple log files...
Press Ctrl+C to stop and generate reports.
2026/10/31 15:13:07 Monitoring: sample-logs/app.log
2026/10/31 15:13:07 Monitoring: sample-logs/server.log

[15:13:07] INFO | app.log | INFO: Application started

[15:13:07] ERROR | app.log | ERROR: Database connection failed

[15:13:07] INFO | app.log | INFO: User logged in

[15:13:07] WARNING | app.log | WARNING: High memory usage

[15:13:07] CRITICAL | app.log | CRITICAL: Server overloaded

[15:13:07] ERROR | app.log | ERROR: Request timeout

[15:13:07] INFO | server.log | INFO: Server running

[15:13:07] ERROR | server.log | ERROR: Disk space low

[15:13:07] INFO | server.log | INFO: Request received

[15:13:07] FAILURE | server.log | FAILURE: Authentication failed

[15:13:07] ERROR | server.log | ERROR: Service unavailable

Stopping LogPulse...
LogPulse stopped.


In [8]:
with open("sample-logs/app.log", "a") as f:
    f.write("ERROR: New database issue\n")
    f.write("CRITICAL: System failure detected\n")
    f.write("FAILURE: Request processing failed\n")

print("New log entries added!")

New log entries added!


In [9]:
import os

print("Generated files:")

for file in ["summary.csv", "summary.json"]:
    if os.path.exists(file):
        print(file)
    else:
        print(file, "not found")

Generated files:
summary.csv
summary.json


In [10]:
import pandas as pd

df = pd.read_csv("summary.csv")
display(df)

,File,Error Type,Count
0,server.log,ALL,3
1,app.log,ALL,3
2,ALL FILES,FAILURE,1
3,ALL FILES,ERROR,4
4,ALL FILES,CRITICAL,1


In [11]:
import json

with open("summary.json", "r") as f:
    report = json.load(f)

print(json.dumps(report, indent=4))

{
    "generated_at": "2026-10-31T14:59:56.402927589Z",
    "total_errors": 6,
    "by_file": {
        "app.log": 3,
        "server.log": 3
    },
    "by_type": {
        "CRITICAL": 1,
        "ERROR": 4,
        "FAILURE": 1
    },
    "by_minute": {
        "2026-10-31 14:54": 6
    }
}


In [12]:
from google.colab import files

files.download("main.go")
files.download("summary.csv")
files.download("summary.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>